# 01 — População Ativa

**Objetivo:** diagnosticar a tabela fonte e construir o universo elegível de clientes por data de referência (snapshots).

**Critérios:**
* Cliente ativo = pelo menos 1 compra nos 5 dias anteriores à data de referência
* Snapshots diários de 2024-05-06 a 2024-05-15
* Unicidade garantida de `customerID` + `reference_date`
* Excluir `cardNumber` (PII) de qualquer uso analítico

**Output:** `workspace.ml_bakehouse.population_active`

In [0]:
# ── Configuração central (idem notebook 00) ──
SOURCE  = "samples.bakehouse.sales_transactions"
CATALOG = "workspace"
SCHEMA  = "ml_bakehouse"
HIST_WINDOW   = 5
REF_START, REF_END = "2024-05-06", "2024-05-15"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

from pyspark.sql import functions as F

df_transactions = spark.table(SOURCE).drop("cardNumber")  # PII fora do projeto
df_transactions.printSchema()
display(df_transactions.limit(10))

In [0]:
# Cobertura temporal, duplicidades e nulos
cobertura = df_transactions.agg(
    F.count("*").alias("total_linhas"),
    F.countDistinct("transactionID").alias("transacoes_unicas"),
    F.countDistinct("customerID").alias("clientes_unicos"),
    F.countDistinct("franchiseID").alias("franquias"),
    F.countDistinct("product").alias("produtos"),
    F.min("dateTime").alias("data_minima"),
    F.max("dateTime").alias("data_maxima"),
)
display(cobertura)

# Nulos por coluna
display(df_transactions.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df_transactions.columns]
))

print("Duplicidades de transactionID:",
      df_transactions.count() - df_transactions.select("transactionID").distinct().count())

In [0]:
# Distribuição diária: volume de transações e clientes ativos por dia
trans = df_transactions.withColumn("tx_date", F.to_date("dateTime"))

display(
    trans.groupBy("tx_date")
    .agg(
        F.count("*").alias("transacoes"),
        F.countDistinct("customerID").alias("clientes_ativos_dia"),
    )
    .orderBy("tx_date")
)
# Observação esperada: 2024-05-17 tem dados apenas até 11h48 — dia INCOMPLETO,
# por isso não é usado nem como referência nem como futuro completo.

## Construção dos snapshots

Para cada data de referência $t$, a população contém os clientes com compra em $[t-5,\, t-1]$. A tabela resultante tem **1 linha por cliente × data** — granularidade da modelagem.

In [0]:
from datetime import date, timedelta

ref_start = date.fromisoformat(REF_START)
ref_end   = date.fromisoformat(REF_END)
refs = spark.createDataFrame(
    [(ref_start + timedelta(days=i),) for i in range((ref_end - ref_start).days + 1)],
    "reference_date DATE",
)

# População ativa: clientes com >=1 compra na janela histórica de cada referência
population = (
    refs.alias("r")
    .join(
        trans.alias("t"),
        on=(F.col("t.tx_date") >= F.col("r.reference_date") - F.lit(HIST_WINDOW))
         & (F.col("t.tx_date") <= F.col("r.reference_date") - F.lit(1)),
        how="inner",
    )
    .groupBy(F.col("r.reference_date"), F.col("t.customerID"))
    .agg(
        F.min("t.tx_date").alias("primeira_compra_janela"),
        F.max("t.tx_date").alias("ultima_compra_janela"),
        F.countDistinct("t.transactionID").alias("compras_historia"),
    )
)

(population.write.mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SCHEMA}.population_active"))

print(f"Salvo: {CATALOG}.{SCHEMA}.population_active")
display(population.orderBy("reference_date", "customerID").limit(10))

In [0]:
# Verificações de qualidade
# 1) Unicidade cliente × data (deve ser zero)
duplicados = (
    population.groupBy("reference_date", "customerID").count().filter("count > 1").count()
)
print("Duplicados cliente×referência (esperado 0):", duplicados)

# 2) Volume por snapshot
display(
    population.groupBy("reference_date")
    .agg(F.count("*").alias("clientes"), F.round(F.avg("compras_historia"), 2).alias("compras_media_5d"))
    .orderBy("reference_date")
)

# 3) Distribuição do nº de compras na janela (histograma)
display(population.groupBy("compras_historia").agg(F.count("*").alias("clientes")).orderBy("compras_historia"))

## Conclusões

* ~285–290 clientes ativos por snapshot (de um universo de 300), com ≈ 3,4 compras na média da janela de 5 dias — população estável e suficiente para o estudo.
* Unicidade `customerID` × `reference_date` garantida: base pronta para receber features e target.
* **Próximo passo:** `02_feature_store` — variáveis comportamentais na mesma janela histórica.